# **Imports**

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split , cross_val_score , RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from category_encoders import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import imblearn
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score , classification_report , accuracy_score , confusion_matrix

In [ ]:
df = pd.read_csv('/kaggle/input/personal-key-indicators-of-heart-disease/2022/heart_2022_no_nans.csv')
df.head(5)

# **Explore the data**

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# describtion for numerical columns
df.describe()

# **Calculate missing values**

In [ ]:
df.isna().sum().sort_values(ascending = False).head(5)

> There is no missing values

# **Feature selection**

In [ ]:
# drop 'State' and 'Sex' columns
df.drop(columns = ['State', 'Sex'], inplace = True)

In [ ]:
# Encoding columns
replacement_dict = {'Yes': 1, 'No': 0}
df['HadHeartAttack'] = df['HadHeartAttack'].replace(replacement_dict)
df['HadAngina'] = df['HadAngina'].replace(replacement_dict)

# Create a new column that will be our target column
df['HeartDisease'] = df['HadHeartAttack'] | df['HadAngina']

# Drop old columns
df.drop(columns = ['HadHeartAttack','HadAngina'], inplace = True)

df.shape

>**Here we create our target column 'HeartDisease' from columns ['HadHeartAttack','HadAngina']**

# # **Data distribution**

# **Numerical columns**

In [ ]:
df_num = df.select_dtypes(include = ['float64', 'int64'])
print(df_num.shape)
df_num.head()

**Plot the distribution for all the numerical features.**

In [ ]:
df_num.hist(figsize=(16, 20), bins=40, xlabelsize=6, ylabelsize=6);

In [ ]:
fig, axes = plt.subplots(nrows=len(df_num.columns) // 2, ncols=2, figsize=(13, 10))

for idx, column in enumerate(df_num.drop(columns = 'HeartDisease')):
    row_idx = idx // 2
    col_idx = idx % 2
    
    sns.kdeplot(df[df["HeartDisease"] == 1][column], alpha=0.5, fill=True, color="#000CEB", label="HeartDisease", ax=axes[row_idx, col_idx])
    sns.kdeplot(df[df["HeartDisease"] == 0][column], alpha=0.5, fill=True, color="#97B9F4", label="Normal", ax=axes[row_idx, col_idx])
    
    axes[row_idx, col_idx].set_xlabel(column)
    axes[row_idx, col_idx].set_ylabel("Frequency")
    axes[row_idx, col_idx].set_title(f"{column} Distribution over Heart Disease")
    axes[row_idx, col_idx].legend()

plt.tight_layout()
plt.show()

# **Categorical columns**

**Check for high and low cardinality**

In [ ]:
df_cat = df.select_dtypes('object')
df_cat.nunique().sort_values()

# **Outliers**

In [ ]:
fig, axes = plt.subplots(nrows=len(df_num.columns) // 2, ncols=2, figsize=(16, 20))

for idx, column in enumerate(df_num.drop(columns = 'HeartDisease')):
    row_idx = idx // 2
    col_idx = idx % 2
    
    sns.boxenplot( x='HeartDisease' , y= column , data=df, ax=axes[row_idx, col_idx])
    
    axes[row_idx, col_idx].set_xlabel("Heart Disease")
    axes[row_idx, col_idx].set_ylabel(column)
    axes[row_idx, col_idx].set_title(f"{column} Distribution")

plt.tight_layout()
plt.show()

> **There is no many outliers**

# **Multicollinearity**

In [ ]:
corr = df_num.drop(columns= 'HeartDisease').corr()
fig , ax = plt.subplots(figsize=(15 , 10))
sns.heatmap(corr ,annot= True , ax=ax , cmap= 'Greens');

**Now we will search for columns that has a correlation more than 70% and drop one of them with the condition that the correlation with the target column (HeartDisease) is smaller than another column**

In [ ]:
# check the correlation for columns => WeightInKilograms & BMI with the target
print(f"Correlation between WeightInKilograms and BMI :{df['WeightInKilograms'].corr(df['BMI'])}")

print(f"Correlation between WeightInKilograms and the target :{df['WeightInKilograms'].corr(df['HeartDisease'])}")

print(f"Correlation between BMI and the target :{df['BMI'].corr(df['HeartDisease'])}")

In [ ]:
# drop 'WeightInKilograms' column
df.drop(columns = 'WeightInKilograms', inplace = True)

# **Encoding categorical data**

In [ ]:
#this for Dummy calsifier 
dfc = df

In [ ]:
df_categorical=df_cat.columns
df_categorical

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[('onehot', OneHotEncoder(), df_categorical)],
    remainder='passthrough')

X_transformed = preprocessor.fit_transform(df.drop(columns = 'HeartDisease'))

# **Splitting data for train and test**

In [ ]:
target = df['HeartDisease']

X_train , X_test , y_train , y_test = train_test_split(X_transformed ,target ,test_size=0.2 , random_state=42 )
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# **Target balance**

In [ ]:
heart_disease_column = df.HeartDisease.value_counts()

# pie chart for target column
plt.pie(heart_disease_column, labels = heart_disease_column.index, autopct="%1.1f%%", explode = [0,0.1], colors = ["#0142F4","#00EBB5"])
plt.title("distribution of Heart Disease")
plt.axis("equal")
plt.show()

> **so we can see that data is imbalanced**

# **Resampling data**

**The best technique in resampling is to merge between over_sampling and under_sampling Because this means you do not lose a lot of features values that can make your model better and the duplication in over_sampling tries to make a model prediction balanced between target labels.**

In [ ]:
df.HeartDisease.value_counts()

In [ ]:
over = SMOTE(sampling_strategy = 1)
under = RandomUnderSampler(sampling_strategy = 0.1)

X_train_resampled, y_train_resampled = under.fit_resample(X_train, y_train)
X_train_resampled, y_train_resampled = over.fit_resample(X_train_resampled, y_train_resampled)
Counter(y_train_resampled)

In [ ]:
print("X_train after resampling :", X_train_resampled.shape)
print("y_train after resampling :", y_train_resampled.shape)

# **Baseline**

In [ ]:
dummy_classifier = DummyClassifier(strategy = 'most_frequent') 
dummy_classifier.fit(X_train, y_train) 
y_pred = dummy_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline Model Accuracy: {accuracy:.4f}")

# **Modeling**

In [ ]:
def train(classifier,x_train,y_train,x_test,y_test):
    
    classifier.fit(x_train,y_train)
    prediction = classifier.predict(x_test)
    cv = RepeatedStratifiedKFold(n_splits = 10,n_repeats = 3,random_state = 1)
    print("Cross Validation Score : ",'{0:.2%}'.format(cross_val_score(classifier,x_train,y_train,cv = cv,scoring = 'roc_auc').mean()))
    

def model_evaluation(classifier,x_test,y_test):
    
    # Confusion Matrix
    cm = confusion_matrix(y_test,classifier.predict(x_test))
    names = ['True Neg','False Pos','False Neg','True Pos']
    counts = [value for value in cm.flatten()]
    percentages = ['{0:.2%}'.format(value) for value in cm.flatten()/np.sum(cm)]
    labels = [f'{v1}\n{v2}\n{v3}' for v1, v2, v3 in zip(names,counts,percentages)]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cm,annot = labels,cmap = 'Greens',fmt ='')
    
    # Classification Report
    print(classification_report(y_test,classifier.predict(x_test)))

# Random forest

In [ ]:
rf_classifier = make_pipeline(
    OneHotEncoder(use_cat_names = True),
    MinMaxScaler(),
    RandomForestClassifier(n_estimators=10, random_state=42)
)

**The model with resampling**

In [ ]:
train(rf_classifier, X_train_resampled, y_train_resampled, X_test, y_test)
model_evaluation(rf_classifier, X_test, y_test)

**The model without resampling**

In [ ]:
train(rf_classifier, X_train, y_train, X_test, y_test)
model_evaluation(rf_classifier, X_test, y_test)

# **conclusion**

**As we can see, there is a big difference between the accuracy in the two models**

![](https://i.cbc.ca/1.5359228.1577206958!/fileImage/httpImage/image.jpg_gen/derivatives/16x9_620/smudge-the-viral-cat.jpg)